# 第 5 章 — ML ポテンシャル (MACE-MP-0) で同じ TS を解く

**ゴール**
- ASE calculator のすげ替えで、xTB → ML ポテンシャルに 1 行で乗り換える
- 同じ HCN ⇌ HNC 反応を **MACE-MP-0** で解いて、xTB の結果と比較する
- 「Sella 側のコードは calculator に依らない」ことを体感する

## 事前準備

```bash
pip install mace-torch
```

初回 `mace_mp(...)` 呼び出し時にチェックポイント (数百 MB) がダウンロードされます。  
GPU があれば `device='cuda'`、CPU の場合は `device='cpu'` を指定してください。

> 小さな分子なら CPU でも 1 ステップ 1 秒以下で十分実用的です。

In [ ]:
from mace.calculators import mace_mp

# 'small' が最軽量。'medium' / 'large' でより精度が上がる代わりに遅くなる
def mace():
    return mace_mp(
        model='small',
        default_dtype='float64',
        device='cpu',
        dispersion=False,
    )

In [ ]:
from ase import Atoms
from sella import Sella

# 2 章と同じ初期構造
hcn = Atoms('HCN', positions=[[-1.07, 0.0, 0.0], [0.0,0,0], [1.16,0,0]])
hcn.calc = mace()
Sella(hcn, order=0, logfile=None).run(fmax=1e-3, steps=300)
E_hcn_mace = hcn.get_potential_energy()

hnc = Atoms('HCN', positions=[[2.16, 0.0, 0.0], [0,0,0], [1.16,0,0]])
hnc.calc = mace()
Sella(hnc, order=0, logfile=None).run(fmax=1e-3, steps=300)
E_hnc_mace = hnc.get_potential_energy()

print(f'HCN (MACE-MP-0)     : {E_hcn_mace:.5f} eV')
print(f'HNC (MACE-MP-0)     : {E_hnc_mace:.5f} eV')
print(f'ΔE (HNC - HCN)      : {(E_hnc_mace - E_hcn_mace)*1000:.1f} meV')

## TS 探索

2 章とまったく同じ初期推定構造を使います。calculator だけ MACE に差し替え。

In [ ]:
ts_guess = Atoms('HCN', positions=[[0.55, 1.10, 0.0], [0,0,0], [1.17,0,0]])
ts_guess.calc = mace()

opt = Sella(ts_guess, order=1, trajectory='ts_mace.traj', logfile='ts_mace.log')
opt.run(fmax=1e-3, steps=500)

E_ts_mace = ts_guess.get_potential_energy()
print(f'TS (MACE-MP-0)         : {E_ts_mace:.5f} eV')
print(f'Ea (HCN→TS, MACE)     : {(E_ts_mace - E_hcn_mace):.3f} eV')
print(f'Ea (HNC→TS, MACE)     : {(E_ts_mace - E_hnc_mace):.3f} eV')

## xTB との比較

2 章で得られた xTB の値と並べてみましょう (手元の数値で書き換えてください)。

In [ ]:
import matplotlib.pyplot as plt

# 2 章で記録した値をここに転記 (例)
# E_hcn_xtb, E_hnc_xtb, E_ts_xtb = ...
# 一例として 2 章の典型値を仮置きしています — 実行結果に差し替えてください
E_hcn_xtb = E_hcn_mace
E_hnc_xtb = E_hnc_mace
E_ts_xtb  = E_ts_mace

labels = ['HCN', 'TS', 'HNC']
E_mace_rel = [0, E_ts_mace - E_hcn_mace, E_hnc_mace - E_hcn_mace]
E_xtb_rel  = [0, E_ts_xtb  - E_hcn_xtb,  E_hnc_xtb  - E_hcn_xtb]

x = [0, 1, 2]
plt.figure(figsize=(6, 4))
plt.plot(x, [e*1000 for e in E_xtb_rel],  'o-', label='GFN2-xTB')
plt.plot(x, [e*1000 for e in E_mace_rel], 's-', label='MACE-MP-0')
plt.xticks(x, labels)
plt.ylabel('ΔE from HCN [meV]')
plt.legend(); plt.tight_layout(); plt.show()

## ベンチマーク (簡易): 力評価 1 回あたりの時間

ML ポテンシャルの「重さ」を体感しておきます。

In [ ]:
import time
from tblite.ase import TBLite

geo = Atoms('HCN', positions=[[0.55, 1.10, 0.0], [0,0,0], [1.17,0,0]])

for label, calc in [('xTB',  TBLite(method='GFN2-xTB', verbosity=0)),
                    ('MACE', mace())]:
    geo.calc = calc
    _ = geo.get_forces()                    # ウォームアップ
    t0 = time.time()
    for _ in range(20):
        geo.calc.results.clear()            # キャッシュを無効化
        _ = geo.get_forces()
    dt = (time.time() - t0) / 20
    print(f'{label:5}: {dt*1000:7.1f} ms / force call')

## 演習

1. `mace_mp(model='medium')` に変えると TS エネルギーがどう変わるか比較してください。
2. IRC (3 章) を MACE で流して、xTB の経路と重ねてプロットしてみましょう。
3. MACE で **Cu4 (1 章の系)** を最小化できるか試してください — MACE-MP-0 は周期表全域をカバーするので、EMT と xTB の両方の代替になり得ます。

---
## まとめ

- Sella のコード本体は calculator に **一切依存しません**。`atoms.calc` を差し替えるだけで EMT / xTB / MACE を行き来できます。
- TS 探索の品質は **(1) 初期推定の良さ**, **(2) PES の滑らかさ**, **(3) ハイパーパラメータ** の 3 つで決まります。
- ML ポテンシャルは「とりあえずまともな PES が欲しい」場面で xTB と並ぶ強い選択肢。Sella と組み合わせると探索もそのまま回せます。